# 03 · Evaluación experimental del diseño

**Entregable 4** · Bases de Datos II · PanTEon Database v1.6.1

Rama: `develop`

---

## Objetivo (guía, §4.4)

Comparar **dos estrategias de almacenamiento** de los perfiles de *k*-mers y decidir cuál
conviene según el patrón de uso.

| | Estrategia A | Estrategia B |
|---|---|---|
| Documento | Uno por secuencia y por *k* | Uno por secuencia, con los seis perfiles |
| Colección | `kmers` (1.439.676 docs) | `kmers_estrategia_B` (239.946 docs) |
| Perfil puntual | documento propio | arreglo `perfiles_k` |

## Protocolo

- Se parte de `kmers` ya cargado por el notebook 01.
- Antes de la línea base se **eliminan todos los índices secundarios** de la colección.
- Cada consulta se mide con un calentamiento descartado y **tres repeticiones**; se
  reporta la **mediana**.
- Se usa `explain` con `verbosity: "executionStats"` y se registran `executionTimeMillis`,
  `totalDocsExamined` y `totalKeysExamined`.
- Al terminar, la estrategia A queda con los índices de producción
  (`uq_kmers_id_seq_k` e `ix_kmers_k_valid_windows`), con los mismos nombres y opciones de
  `database/indices.js`.

## Métricas obligatorias

- [x] Tiempo de carga (notebook 01 para A; construcción de B en este notebook).
- [x] Espacio en disco con `collStats` (`storageSize`, `totalIndexSize`, `avgObjSize`).
- [x] Tiempo de 3 consultas representativas, con y sin índice.
- [x] Tabla comparativa y gráficos de barras.


## 1. Entorno y conexión

Se registran sistema operativo, número de CPUs y versión de MongoDB. El resultado se cita
en la sección III-D del artículo.


In [ ]:
import json
import os
import platform
from statistics import median
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "panteon_db"
REPETICIONES = 3

cliente = MongoClient(MONGO_URI)
db = cliente[DB_NAME]
col_a = db["kmers"]
col_b = db["kmers_estrategia_B"]

print("Sistema:  ", platform.platform())
print("CPUs:     ", os.cpu_count())
print("MongoDB:  ", db.command("buildInfo")["version"])
print("Docs en A:", col_a.estimated_document_count())


## 2. Utilidades de medición

`medir` hace un calentamiento y tres corridas por consulta, y devuelve la mediana de los
tres contadores. `borrar_indices_secundarios` deja la colección solo con el índice `_id`,
que es lo que define la línea base.


In [ ]:
def borrar_indices_secundarios(coleccion):
    """Elimina todos los índices excepto el de _id."""
    for nombre in list(coleccion.index_information()):
        if nombre != "_id_":
            coleccion.drop_index(nombre)


def medir(coleccion, filtro):
    """Mediana de tres corridas de explain con executionStats."""
    coleccion.find(filtro).limit(1).to_list(length=1)  # calentamiento
    corridas = []
    for _ in range(REPETICIONES):
        plan = db.command(
            {
                "explain": {"find": coleccion.name, "filter": filtro},
                "verbosity": "executionStats",
            }
        )["executionStats"]
        corridas.append(
            {
                "ms": plan["executionTimeMillis"],
                "docs": plan["totalDocsExamined"],
                "llaves": plan["totalKeysExamined"],
                "n": plan["nReturned"],
            }
        )
    return {
        "ms": median(c["ms"] for c in corridas),
        "docs": median(c["docs"] for c in corridas),
        "llaves": median(c["llaves"] for c in corridas),
        "n": corridas[-1]["n"],
        "corridas": [c["ms"] for c in corridas],
    }


def stats(coleccion):
    s = db.command("collstats", coleccion.name)
    return {
        "documentos": s["count"],
        "storage_mb": s["storageSize"] / (1024 ** 2),
        "indices_mb": s["totalIndexSize"] / (1024 ** 2),
        "avg_obj": s["avgObjSize"],
    }


def imprimir(nombre, r):
    print(
        f"  {nombre}: mediana {r['ms']:.0f} ms | docs {r['docs']:.0f} | "
        f"llaves {r['llaves']:.0f} | devueltos {r['n']} | corridas {r['corridas']}"
    )


## 3. Consultas representativas

| Consulta | Patrón | A | B |
|---|---|---|---|
| Q1 | Perfil puntual de una secuencia para un *k* | `{id_seq, k}` | `{id_seq, perfiles_k.k}` |
| Q2 | Perfil completo de una secuencia | `{id_seq}` | `{id_seq}` |
| Q3 | Filtro por tamaño de la secuencia | `{k: 6, valid_windows: {$gt: 1500}}` | `$elemMatch` sobre `perfiles_k` |


## 4. Estrategia A · línea base sin índices

Se eliminan los índices secundarios antes de medir. Sin este paso, una corrida anterior
deja `{id_seq, k}` y `{k, valid_windows}` creados y las consultas puntuales responden en
0 ms, con lo que la comparación no mide nada.


In [ ]:
q1 = {"id_seq": "PDB00000001", "k": 6}
q2 = {"id_seq": "PumCon-1.11"}
q3 = {"k": 6, "valid_windows": {"$gt": 1500}}

print("Estrategia A · sin índices secundarios")
borrar_indices_secundarios(col_a)
base_a = {
    "Q1": medir(col_a, q1),
    "Q2": medir(col_a, q2),
    "Q3": medir(col_a, q3),
}
for nombre, r in base_a.items():
    imprimir(nombre, r)


## 5. Estrategia A · con índices de producción

Los nombres y las opciones coinciden con `database/indices.js`, de modo que ejecutar ese
script después no produce conflicto y la unicidad de `{id_seq, k}` queda garantizada.


In [ ]:
col_a.create_index(
    [("id_seq", 1), ("k", 1)], unique=True, name="uq_kmers_id_seq_k"
)
col_a.create_index(
    [("k", 1), ("valid_windows", 1)], name="ix_kmers_k_valid_windows"
)

print("Estrategia A · con índices")
ind_a = {
    "Q1": medir(col_a, q1),
    "Q2": medir(col_a, q2),
    "Q3": medir(col_a, q3),
}
for nombre, r in ind_a.items():
    imprimir(nombre, r)


## 6. Estrategia B · construcción

La colección B se genera con una agregación `$out` sobre A. Se cronometra la operación
completa porque es el costo de pasar de una representación a la otra.


In [ ]:
col_b.drop()
pipeline_b = [
    {
        "$group": {
            "_id": "$sequence_id",
            "id_seq": {"$first": "$id_seq"},
            "perfiles_k": {
                "$push": {
                    "k": "$k",
                    "valid_windows": "$valid_windows",
                    "counts": "$counts",
                }
            },
        }
    },
    {"$out": "kmers_estrategia_B"},
]

inicio = perf_counter()
list(col_a.aggregate(pipeline_b, allowDiskUse=True))
t_reorganizacion_b = perf_counter() - inicio
print(f"Construcción de B: {t_reorganizacion_b:.1f} s")
print(f"Documentos en B:   {col_b.estimated_document_count()}")


## 7. Estrategia B · sin índices


In [ ]:
q1_b = {"id_seq": "PDB00000001", "perfiles_k.k": 6}
q2_b = {"id_seq": "PumCon-1.11"}
q3_b = {"perfiles_k": {"$elemMatch": {"k": 6, "valid_windows": {"$gt": 1500}}}}

print("Estrategia B · sin índices secundarios")
borrar_indices_secundarios(col_b)
base_b = {
    "Q1": medir(col_b, q1_b),
    "Q2": medir(col_b, q2_b),
    "Q3": medir(col_b, q3_b),
}
for nombre, r in base_b.items():
    imprimir(nombre, r)


## 8. Estrategia B · con índices


In [ ]:
col_b.create_index([("id_seq", 1)], name="ix_B_id_seq")
col_b.create_index(
    [("perfiles_k.k", 1), ("perfiles_k.valid_windows", 1)],
    name="ix_B_perfiles_k",
)

print("Estrategia B · con índices")
ind_b = {
    "Q1": medir(col_b, q1_b),
    "Q2": medir(col_b, q2_b),
    "Q3": medir(col_b, q3_b),
}
for nombre, r in ind_b.items():
    imprimir(nombre, r)


## 9. Espacio en disco y tiempo de carga

El tiempo de carga de A lo escribe el notebook 01 en `results/carga_a.json`; el de B es la
suma de ese tiempo más la construcción por `$out`. Si el archivo no existe, la columna
queda vacía y hay que ejecutar el notebook 01 antes.


In [ ]:
stats_a = stats(col_a)
stats_b = stats(col_b)

carga_a = None
try:
    with open("../results/carga_a.json", encoding="utf-8") as fh:
        carga_a = json.load(fh)["segundos"]
except FileNotFoundError:
    print(
        "Aviso: falta results/carga_a.json. "
        "Ejecuta el notebook 01 para medir la carga de A."
    )

resumen = pd.DataFrame(
    [
        {"estrategia": "A · un doc por k", **stats_a, "carga_s": carga_a},
        {
            "estrategia": "B · seis perfiles por doc",
            **stats_b,
            "carga_s": (carga_a + t_reorganizacion_b) if carga_a else None,
        },
    ]
)
display(resumen)
os.makedirs("../results", exist_ok=True)
resumen.to_csv("../results/benchmark_espacio.csv", index=False)


## 10. Tabla comparativa de consultas


In [ ]:
filas = []
for etiqueta, base, con in (("A", base_a, ind_a), ("B", base_b, ind_b)):
    for consulta in ("Q1", "Q2", "Q3"):
        filas.append(
            {
                "estrategia": etiqueta,
                "consulta": consulta,
                "sin_indice_ms": base[consulta]["ms"],
                "con_indice_ms": con[consulta]["ms"],
                "docs_sin": base[consulta]["docs"],
                "docs_con": con[consulta]["docs"],
                "devueltos_sin": base[consulta]["n"],
                "devueltos_con": con[consulta]["n"],
                "llaves_sin": base[consulta]["llaves"],
                "llaves_con": con[consulta]["llaves"],
            }
        )

benchmark = pd.DataFrame(filas)
display(benchmark)
os.makedirs("../results", exist_ok=True)
benchmark.to_csv("../results/benchmark_tiempos.csv", index=False)


## 11. Gráficos

Se guardan en `paper/figures/` en PNG a 300 dpi y PDF vectorial. El eje de tiempo va en
escala logarítmica porque las consultas con índice bajan a 0–1 ms y las que hacen *scan*
superan los 3.000 ms.


In [ ]:
consultas = ["Q1", "Q2", "Q3"]
x = list(range(len(consultas)))
series = [
    ("A · sin índice", [base_a[q]["ms"] for q in consultas]),
    ("A · con índice", [ind_a[q]["ms"] for q in consultas]),
    ("B · sin índice", [base_b[q]["ms"] for q in consultas]),
    ("B · con índice", [ind_b[q]["ms"] for q in consultas]),
]

fig, ax = plt.subplots(figsize=(7, 3.4))
ancho = 0.2
for i, (etiqueta, valores) in enumerate(series):
    ax.bar([p + (i - 1.5) * ancho for p in x], valores, width=ancho, label=etiqueta)

ax.set_xticks(x)
ax.set_xticklabels(consultas)
ax.set_ylabel("Mediana de executionTimeMillis (ms)")
ax.set_yscale("log")
ax.set_title("Tiempo de las consultas representativas")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig("../paper/figures/benchmark_tiempos.png", dpi=300)
fig.savefig("../paper/figures/benchmark_tiempos.pdf")
plt.show()


In [ ]:
etiquetas = ["A · un doc por k", "B · seis perfiles por doc"]
datos = [stats_a["storage_mb"], stats_b["storage_mb"]]
indices = [stats_a["indices_mb"], stats_b["indices_mb"]]

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.bar(etiquetas, datos, label="Datos")
ax.bar(etiquetas, indices, bottom=datos, label="Índices")
ax.set_ylabel("MB")
ax.set_title("Espacio en disco por estrategia")
ax.legend()
fig.tight_layout()
fig.savefig("../paper/figures/benchmark_espacio.png", dpi=300)
fig.savefig("../paper/figures/benchmark_espacio.pdf")
plt.show()


## 12. Lectura de resultados

- **Tiempo.** Comparar cada consulta con y sin índice dentro de la misma estrategia; la
  diferencia entre A y B solo es interpretable con el mismo protocolo. Q1 y Q2 son
  búsquedas puntuales y deberían beneficiarse de `{id_seq, k}` o `{id_seq}`. Q3 es una
  consulta de rango sobre un atributo poco selectivo: si el índice no reduce los
  documentos examinados, puede añadir un `FETCH` y empeorar el tiempo.
- **Qué mirar si Q3 no mejora.** No concluir que “el índice no sirve”: revisar
  `totalDocsExamined` y `totalKeysExamined`. Si el índice examina casi los mismos
  documentos que el recorrido completo, el filtro no es selectivo; si examina pocos pero
  el tiempo sube, el costo está en el `FETCH`, sobre todo en B, donde cada documento pesa
  unos 33,5 KB.
- **Espacio.** `storageSize` mide datos y `totalIndexSize` mide índices; el total es la
  suma. B repite menos metadatos por perfil, así que suele ocupar menos.
- **Qué estrategia conviene.** A favor de A: el perfil puntual no arrastra los otros cinco
  *k*. A favor de B: menos documentos, menos índices y una lectura por secuencia.
- **Limpieza.** `kmers_estrategia_B` ocupa varios GB. Si no se va a volver a consultar:
  `db.kmers_estrategia_B.drop()`.

Las tablas se guardan en `results/benchmark_tiempos.csv` y `results/benchmark_espacio.csv`;
las figuras, en `paper/figures/benchmark_tiempos.*` y `benchmark_espacio.*`. La Tabla VII
del artículo solo se reemplaza con una corrida completa y válida; la corrida preliminar no
se usa para conclusiones de tiempo.
